In [1]:
# loaded  dataset from the hugging face
from datasets import load_dataset

dataset = load_dataset("prithvi1029/medquad-medical-qa")

df = dataset["train"].to_pandas()

print(df.shape)
print(df.columns)
print(df.head())

(16407, 3)
Index(['qtype', 'Question', 'Answer'], dtype='object')
             qtype                                           Question  \
0   susceptibility  Who is at risk for Lymphocytic Choriomeningiti...   
1         symptoms  What are the symptoms of Lymphocytic Choriomen...   
2   susceptibility  Who is at risk for Lymphocytic Choriomeningiti...   
3  exams and tests  How to diagnose Lymphocytic Choriomeningitis (...   
4        treatment  What are the treatments for Lymphocytic Chorio...   

                                              Answer  
0  LCMV infections can occur after exposure to fr...  
1  LCMV is most commonly recognized as causing ne...  
2  Individuals of all ages who come into contact ...  
3  During the first phase of the disease, the mos...  
4  Aseptic meningitis, encephalitis, or meningoen...  


In [2]:
print(df.isnull().sum())
print(df.duplicated().sum())
print(df["Question"].iloc[0])
print(df["Answer"].iloc[0])

qtype       0
Question    0
Answer      0
dtype: int64
48
Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?
LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.


In [3]:
# Remove duplicate Question + Answer pairs
df = df.drop_duplicates(
    subset=["Question", "Answer"]
).reset_index(drop=True)

print("New shape:", df.shape)
print("Duplicates:", df.duplicated(subset=["Question", "Answer"]).sum())

New shape: (16359, 3)
Duplicates: 0


In [4]:
#cleaning whitespace

df["Question"] = df["Question"].str.strip()
df["Answer"] = df["Answer"].str.strip()

print(df.head())

             qtype                                           Question  \
0   susceptibility  Who is at risk for Lymphocytic Choriomeningiti...   
1         symptoms  What are the symptoms of Lymphocytic Choriomen...   
2   susceptibility  Who is at risk for Lymphocytic Choriomeningiti...   
3  exams and tests  How to diagnose Lymphocytic Choriomeningitis (...   
4        treatment  What are the treatments for Lymphocytic Chorio...   

                                              Answer  
0  LCMV infections can occur after exposure to fr...  
1  LCMV is most commonly recognized as causing ne...  
2  Individuals of all ages who come into contact ...  
3  During the first phase of the disease, the mos...  
4  Aseptic meningitis, encephalitis, or meningoen...  


In [5]:
# Check number of samples
print("Total samples:", len(df))

# Check columns
print(df.columns)

# Check question lengths
df["question_len"] = df["Question"].str.split().str.len()

# Check answer lengths
df["answer_len"] = df["Answer"].str.split().str.len()

print(df[["question_len", "answer_len"]].describe())

Total samples: 16359
Index(['qtype', 'Question', 'Answer'], dtype='object')
       question_len    answer_len
count  16359.000000  16359.000000
mean       8.214561    200.366587
std        2.381010    246.873458
min        3.000000      1.000000
25%        6.000000     71.000000
50%        8.000000    138.000000
75%       10.000000    251.000000
max       27.000000   4281.000000


In [6]:
print(
    df.sort_values("answer_len", ascending=False)
      [["Question", "Answer", "answer_len"]]
      .head(5)
)

                                               Question  \
2957        What are the treatments for Breast Cancer ?   
2989  What are the treatments for Childhood Soft Tis...   
2779          What is (are) Childhood Vascular Tumors ?   
4393  What are the treatments for Sickle Cell Disease ?   
2996  What are the treatments for Childhood Rhabdomy...   

                                                 Answer  answer_len  
2957  Key Points\n                    - There are di...        4281  
2989  Key Points\n                    - There are di...        3708  
2779  Key Points\n                    - Childhood va...        3692  
4393  Health Maintenance To Prevent Complications\n ...        3594  
2996  Key Points\n                    - There are di...        3461  


In [7]:
#We convert each Question–Answer pair into a format that Qwen understands for instruction tuning.
def format_instruction(row):
    return f"""### Instruction:
Answer the medical question accurately and concisely.

### Question:
{row['Question']}

### Response:
{row['Answer']}"""

df["text"] = df.apply(format_instruction, axis=1)

In [8]:
print(df["text"].iloc[0])

### Instruction:
Answer the medical question accurately and concisely.

### Question:
Who is at risk for Lymphocytic Choriomeningitis (LCM)? ?

### Response:
LCMV infections can occur after exposure to fresh urine, droppings, saliva, or nesting materials from infected rodents.  Transmission may also occur when these materials are directly introduced into broken skin, the nose, the eyes, or the mouth, or presumably, via the bite of an infected rodent. Person-to-person transmission has not been reported, with the exception of vertical transmission from infected mother to fetus, and rarely, through organ transplantation.


In [9]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 13087
Validation: 1636
Test: 1636


In [10]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

In [11]:
print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['qtype', 'Question', 'Answer', 'question_len', 'answer_len', 'text', '__index_level_0__'],
    num_rows: 13087
})
Dataset({
    features: ['qtype', 'Question', 'Answer', 'question_len', 'answer_len', 'text', '__index_level_0__'],
    num_rows: 1636
})
Dataset({
    features: ['qtype', 'Question', 'Answer', 'question_len', 'answer_len', 'text', '__index_level_0__'],
    num_rows: 1636
})


In [12]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token

In [13]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=2048,
        padding="max_length"
    )

In [14]:
tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/13087 [00:00<?, ? examples/s]

In [15]:
tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/1636 [00:00<?, ? examples/s]

In [16]:
tokenized_test = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/1636 [00:00<?, ? examples/s]

In [17]:
print(tokenized_train[0].keys())
print(tokenized_train[0]["input_ids"][:20])
print(tokenized_train[0]["attention_mask"][:20])

dict_keys(['qtype', 'Question', 'Answer', 'question_len', 'answer_len', 'text', '__index_level_0__', 'input_ids', 'attention_mask'])
[14374, 29051, 510, 16141, 279, 6457, 3405, 29257, 323, 3529, 285, 974, 382, 14374, 15846, 510, 3838, 525, 279, 13495]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [18]:
#!pip install -U transformers peft bitsandbytes accelerate

In [19]:
!pip uninstall -y transformers peft accelerate
!pip install -U transformers peft accelerate bitsandbytes

Found existing installation: transformers 5.17.0
Uninstalling transformers-5.17.0:
  Successfully uninstalled transformers-5.17.0
Found existing installation: peft 0.21.0
Uninstalling peft-0.21.0:
  Successfully uninstalled peft-0.21.0
Found existing installation: accelerate 1.15.0
Uninstalling accelerate-1.15.0:
  Successfully uninstalled accelerate-1.15.0
  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
  Using cached peft-0.21.0-py3-none-any.whl.metadata (15 kB)
  Using cached accelerate-1.15.0-py3-none-any.whl.metadata (19 kB)
Using cached transformers-5.17.0-py3-none-any.whl (12.3 MB)
Using cached peft-0.21.0-py3-none-any.whl (832 kB)
Using cached accelerate-1.15.0-py3-none-any.whl (394 kB)


In [20]:
import transformers
import torch
import peft
import accelerate
import bitsandbytes

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 5.17.0
PyTorch: 2.10.0+cu128
PEFT: 0.21.0
Accelerate: 1.15.0


In [21]:
#Load Qwen in 4-bit

import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [22]:
#Prepare Qwen for QLoRA training
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [23]:
# Create LoRA configuration
#peft --> provides LORA
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ],
    task_type="CAUSAL_LM"
)

In [24]:
#Attach LoRA to Qwen
from peft import get_peft_model

model = get_peft_model(model, lora_config)

In [25]:
model.print_trainable_parameters()

trainable params: 10,092,544 || all params: 7,625,709,056 || trainable%: 0.1323


In [26]:
#Now we connect your MedQuAD dataset → tokenizer → QLoRA Qwen → Trainer → fine-tuning.

def format_instruction(example):
    return {
        "text": f"""### Instruction:
Answer the medical question accurately and concisely.

### Question:
{example['Question']}

### Response:
{example['Answer']}"""
    }

train_dataset = train_dataset.map(format_instruction)
val_dataset = val_dataset.map(format_instruction)

Map:   0%|          | 0/13087 [00:00<?, ? examples/s]

Map:   0%|          | 0/1636 [00:00<?, ? examples/s]

In [27]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=2048,
        padding="max_length"
    )

tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True
)

tokenized_val = val_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/13087 [00:00<?, ? examples/s]

Map:   0%|          | 0/1636 [00:00<?, ? examples/s]

In [28]:
def add_labels(example):
    example["labels"] = example["input_ids"].copy()
    return example

tokenized_train = tokenized_train.map(add_labels)
tokenized_val = tokenized_val.map(add_labels)

Map:   0%|          | 0/13087 [00:00<?, ? examples/s]

Map:   0%|          | 0/1636 [00:00<?, ? examples/s]

In [29]:
#A data collator prepares examples into batches before sending them to the model.
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [30]:
  #Create the training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-medical-qlora",

    num_train_epochs=3,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    fp16=True,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_steps=100,
    save_total_limit=2,

    report_to="none"
)

In [31]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator
)

In [1]:
trainer.train()

NameError: name 'trainer' is not defined

In [ ]:
model.save_pretrained("./qwen-medical-qlora")
tokenizer.save_pretrained("./qwen-medical-qlora")